# 🛒 AI Shopping Assistant — Computer Vision & OCR Engine
### Production-grade detection + text-understanding pipeline

**Pipeline:** `Image → YOLOv8 Detection → Per-object OCR → Structured JSON → NLP Module`

| Stage | Cells | Output artefact |
|---|---|---|
| Environment & config | 2–4 | `config.json`, project scaffold |
| Dataset engineering | 5–9 | cleaned YOLO dataset, `data.yaml`, dataset report |
| Training | 10–11 | `models/best.pt`, checkpoints, training report |
| Evaluation & export | 12–13 | metrics, curves, confusion matrix, ONNX |
| OCR engine | 14 | `utils/ocr_module.py` |
| Integration | 15 | `integration/pipeline.py` |
| QA & delivery | 16–18 | quality gates, inference demo, release bundle |

> Every cell is idempotent and safe to re-run after a runtime restart.

In [ ]:
import importlib
import os
import subprocess
import sys
import time

# numpy is pinned FIRST and alone: easyocr/ultralytics/fiftyone all compile
# against the 1.26 ABI. Installing it later triggers a mandatory restart.
_ORDERED_REQUIREMENTS = [
    ["numpy==1.26.4"],
    ["opencv-python-headless==4.10.0.84"],
    ["ultralytics==8.3.94"],
    ["roboflow==1.1.51"],
    ["easyocr==1.7.2"],
    ["imagehash==4.3.1", "Pillow==10.4.0"],
    ["rich==13.9.4", "tqdm==4.66.5", "psutil==6.0.0", "PyYAML==6.0.2"],
    ["pandas==2.2.2", "matplotlib==3.9.2", "seaborn==0.13.2", "tabulate==0.9.0"],
    ["onnx==1.17.0", "onnxruntime==1.19.2"],
    ["fiftyone==1.0.1"],
]

def _pip(args, attempts=3):
    """pip install with exponential backoff"""
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--no-input", *args]
    for attempt in range(1, attempts + 1):
        proc = subprocess.run(cmd, capture_output=True, text=True)
        if proc.returncode == 0:
            return True, ""
        if attempt < attempts:
            time.sleep(4 * attempt)
    return False, (proc.stderr or "")[-600:]

print("=" * 70)
print(" RE-INSTALLING PINNED DEPENDENCY SET".center(70))
print("=" * 70)

_failures = []
for group in _ORDERED_REQUIREMENTS:
    label = ", ".join(p.split("==")[0] for p in group)
    print(f"  → {label:<48}", end="", flush=True)
    ok, err = _pip(group)
    print("OK" if ok else "FAILED")
    if not ok:
        _failures.append((label, err))

# Ensure clean environment
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "opencv-python"], capture_output=True)

import numpy as _np
print("\n" + "=" * 70)
print(f" STATUS: numpy version {_np.__version__}")
if _np.__version__.startswith("1.26"):
    print(" SUCCESS: Environment is consistent and ready.")
else:
    print(" WARNING: numpy version mismatch. Restart might be needed again.")
print("=" * 70)

                  RE-INSTALLING PINNED DEPENDENCY SET                 
  → numpy                                           OK
  → opencv-python-headless                          OK
  → ultralytics                                     OK
  → roboflow                                        OK
  → easyocr                                         OK
  → imagehash, Pillow                               OK
  → rich, tqdm, psutil, PyYAML                      OK
  → pandas, matplotlib, seaborn, tabulate           OK
  → onnx, onnxruntime                               OK
  → fiftyone                                        OK

 STATUS: numpy version 2.0.2


In [ ]:
# =====================================================================
#  CELL 3 | RUNTIME CORE
#  Structured logging, deterministic seeding, hardware profiling,
#  and a single immutable configuration object used by every later cell.
# =====================================================================
from __future__ import annotations

import dataclasses
import json
import logging
import os
import platform
import random
import shutil
import sys
import time
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import psutil
import torch
from rich.console import Console
from rich.logging import RichHandler
from rich.panel import Panel
from rich.table import Table

CONSOLE = Console(width=110)

def _build_logger() -> logging.Logger:
    logger = logging.getLogger("cv_engine")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.propagate = False
    handler = RichHandler(
        console=CONSOLE,
        rich_tracebacks=True,
        markup=True,
        show_path=False,
        omit_repeated_times=False,
    )
    handler.setFormatter(logging.Formatter("%(message)s", datefmt="%H:%M:%S"))
    logger.addHandler(handler)
    return logger

LOG = _build_logger()

def log_ok(msg: str) -> None:
    LOG.info(f"[bold green]✔[/bold green] {msg}")

def log_step(msg: str) -> None:
    LOG.info(f"[bold cyan]▶[/bold cyan] {msg}")

def log_warn(msg: str) -> None:
    LOG.warning(f"[bold yellow]▲[/bold yellow] {msg}")

def log_err(msg: str) -> None:
    LOG.error(f"[bold red]✘[/bold red] {msg}")

def banner(title: str, subtitle: str = "") -> None:
    body = f"[bold white]{title}[/bold white]"
    if subtitle:
        body += f"\n[dim]{subtitle}[/dim]"
    CONSOLE.print(Panel(body, border_style="cyan", expand=True))

# --------------------------- determinism -----------------------------
GLOBAL_SEED = 42

def set_global_determinism(seed: int = GLOBAL_SEED) -> None:
    """Seed every RNG that can influence dataset order, splits or training."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # cudnn.deterministic is intentionally NOT forced: it halves conv throughput
    # on a T4 and Ultralytics already seeds its dataloader workers.
    torch.backends.cudnn.benchmark = True

set_global_determinism()

# ------------------------ hardware profiling -------------------------
@dataclass(frozen=True)
class HardwareProfile:
    device: str
    gpu_name: str
    gpu_total_gb: float
    cpu_count: int
    ram_total_gb: float
    disk_free_gb: float
    amp_supported: bool
    python_version: str
    torch_version: str

    @property
    def is_gpu(self) -> bool:
        return self.device == "cuda"

def profile_hardware() -> HardwareProfile:
    has_cuda = torch.cuda.is_available()
    gpu_name, gpu_gb = "None (CPU only)", 0.0
    if has_cuda:
        props = torch.cuda.get_device_properties(0)
        gpu_name = props.name
        gpu_gb = round(props.total_memory / 1024 ** 3, 2)
    return HardwareProfile(
        device="cuda" if has_cuda else "cpu",
        gpu_name=gpu_name,
        gpu_total_gb=gpu_gb,
        cpu_count=os.cpu_count() or 2,
        ram_total_gb=round(psutil.virtual_memory().total / 1024 ** 3, 2),
        disk_free_gb=round(shutil.disk_usage("/content").free / 1024 ** 3, 2),
        amp_supported=has_cuda,
        python_version=platform.python_version(),
        torch_version=torch.__version__,
    )

HW = profile_hardware()

# ------------------------- project configuration ---------------------
@dataclass
class ProjectConfig:
    """Single source of truth. Serialised to config.json for reproducibility."""

    root: Path = Path("/content/drive/MyDrive/NTI tasks/AI Shopping Assistant")
    seed: int = GLOBAL_SEED

    # dataset
    target_images_per_class: int = 320
    min_images_per_class: int = 40
    min_resolution: int = 96
    dup_hash_distance: int = 4
    split_ratios: Tuple[float, float, float] = (0.75, 0.15, 0.10)

    # model / training
    model_weights: str = "yolov8s.pt"
    fallback_weights: str = "yolov8n.pt"
    imgsz: int = 640
    epochs: int = 90
    batch: int = 16
    patience: int = 18
    warmup_epochs: float = 3.0
    optimizer: str = "AdamW"
    lr0: float = 0.0015
    lrf: float = 0.01
    weight_decay: float = 0.0005
    freeze_backbone_layers: int = 0
    copy_paste: float = 0.25
    mixup: float = 0.10
    experiment: str = "cv_v1"

    # inference / OCR
    conf_threshold: float = 0.35
    iou_threshold: float = 0.55
    max_detections: int = 12
    ocr_conf_threshold: float = 0.40
    ocr_languages: Tuple[str, ...] = ("en",)

    # derived paths
    def __post_init__(self) -> None:
        r = self.root
        self.dataset_dir = r / "dataset"
        self.raw_dir = self.dataset_dir / "_raw"
        self.staging_dir = self.dataset_dir / "_staging"
        self.final_dir = self.dataset_dir / "grocery_yolo"
        self.cv_dir = r / "cv"
        self.runs_dir = self.cv_dir / "runs"
        self.reports_dir = self.cv_dir / "reports"
        self.logs_dir = self.cv_dir / "logs"
        self.nlp_dir = r / "nlp"
        self.integration_dir = r / "integration"
        self.deployment_dir = r / "deployment"
        self.models_dir = r / "models"
        self.utils_dir = r / "utils"
        self.assets_dir = r / "assets"
        self.exports_dir = self.models_dir / "exports"
        self.data_yaml = self.final_dir / "data.yaml"

    def all_dirs(self) -> List[Path]:
        return [
            self.dataset_dir, self.raw_dir, self.staging_dir, self.final_dir,
            self.cv_dir, self.runs_dir, self.reports_dir, self.logs_dir,
            self.nlp_dir, self.integration_dir, self.deployment_dir,
            self.models_dir, self.utils_dir, self.assets_dir, self.exports_dir,
        ]

    def to_json(self) -> Dict[str, Any]:
        payload = {
            k: (str(v) if isinstance(v, Path) else v)
            for k, v in asdict(self).items()
        }
        payload["hardware"] = asdict(HW)
        return payload

CFG = ProjectConfig()

# ---- hardware-aware auto-tuning (never trust static hyperparameters) ----
if not HW.is_gpu:
    CFG.model_weights = CFG.fallback_weights
    CFG.imgsz, CFG.batch, CFG.epochs = 480, 8, 35
    log_warn("No CUDA device. Downgraded to yolov8n / 480px / 35 epochs so the "
             "notebook still completes end-to-end on CPU.")
elif HW.gpu_total_gb < 12:
    CFG.batch = 12
    log_warn(f"GPU has {HW.gpu_total_gb} GB VRAM → batch reduced to 12 to "
             "pre-empt CUDA OOM.")

table = Table(title="RUNTIME PROFILE", header_style="bold cyan", expand=True)
table.add_column("Property"); table.add_column("Value", style="bold")
for k, v in asdict(HW).items():
    table.add_row(k.replace("_", " ").title(), str(v))
table.add_row("Model", CFG.model_weights)
table.add_row("Image size / Batch", f"{CFG.imgsz} / {CFG.batch}")
table.add_row("Epochs / Patience", f"{CFG.epochs} / {CFG.patience}")
CONSOLE.print(table)
log_ok("Runtime core initialised and deterministically seeded.")

                                               RUNTIME PROFILE                                                
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Property                                                      ┃ Value                                      ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Device                                                        │ cuda                                       │
│ Gpu Name                                                      │ Tesla T4                                   │
│ Gpu Total Gb                                                  │ 14.56                                      │
│ Cpu Count                                                     │ 2                                          │
│ Ram Total Gb                                                  │ 12.67                                      │
│ Disk Free Gb                                                  │ 64.74                                      │
│ Amp Supported                                                 │ True                                       │
│ Python Version                                                │ 3.12.13                                    │
│ Torch Version                                                 │ 2.11.0+cu128                               │
│ Model                                                         │ yolov8s.pt                                 │
│ Image size / Batch                                            │ 640 / 16                                   │
│ Epochs / Patience                                             │ 90 / 18                                    │
└───────────────────────────────────────────────────────────────┴────────────────────────────────────────────┘

08:38:45 INFO     ✔ Runtime core initialised and deterministically seeded.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: Mountpoint must not already contain files

In [ ]:
from pathlib import Path

# المسار ده هيدخل على الـ Shortcut ويقرأ داتا الحساب القديم فوراً!
DRIVE_BASE_DIR = Path("/content/drive/MyDrive/NTI tasks/AI Shopping Assistant/Roboflow_Data")

if DRIVE_BASE_DIR.exists():
    print("✅ العظمة! الكولاب الجديد قرا داتا الحساب القديم بنجاح الجاهزة للتدريب.")
else:
    print("⚠️ اخلع الـ Shortcut واتأكد إنك عملت Add shortcut to Drive في الحساب الجديد.")

✅ العظمة! الكولاب الجديد قرا داتا الحساب القديم بنجاح الجاهزة للتدريب.


In [ ]:
# =====================================================================
#  CELL 4 | PERSISTENT WORKSPACE
#  Mounts Drive (idempotent), builds the exact folder contract from the
#  project guide, guards disk space, and attaches a file log handler.
# =====================================================================
DRIVE_MOUNTPOINT = Path("/content/drive")

def mount_drive(max_attempts: int = 2) -> bool:
    """Mount Drive only if not already mounted; tolerate non-Colab runtimes."""
    if (DRIVE_MOUNTPOINT / "MyDrive").exists():
        log_ok("Google Drive already mounted.")
        return True
    try:
        from google.colab import drive as _colab_drive
    except ImportError:
        log_warn("Not running on Colab — using local filesystem instead.")
        return False
    for attempt in range(1, max_attempts + 1):
        try:
            _colab_drive.mount(str(DRIVE_MOUNTPOINT), force_remount=False)
            log_ok("Google Drive mounted.")
            return True
        except Exception as exc:
            log_warn(f"Mount attempt {attempt} failed: {exc}")
            time.sleep(3)
    return False

if not mount_drive():
    # Graceful degradation: keep the identical folder contract locally so no
    # later cell needs a code change.
    CFG.root = Path("/content/AI Shopping Assistant")
    CFG.__post_init__()
    log_warn(f"Drive unavailable. Project root relocated to {CFG.root}")

for directory in CFG.all_dirs():
    directory.mkdir(parents=True, exist_ok=True)

# Persistent file logging survives cell re-runs and disconnects.
_file_handler = logging.FileHandler(CFG.logs_dir / "cv_engine.log", mode="a")
_file_handler.setFormatter(
    logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s")
)
if not any(isinstance(h, logging.FileHandler) for h in LOG.handlers):
    LOG.addHandler(_file_handler)

# ---- disk guard: Drive quota exhaustion mid-training corrupts weights ----
_free_root = shutil.disk_usage(str(CFG.root)).free / 1024 ** 3
if _free_root < 6:
    log_warn(f"Only {_free_root:.1f} GB free on the project volume. Dataset "
             "targets will be scaled down to protect training.")
    CFG.target_images_per_class = min(CFG.target_images_per_class, 150)

(CFG.root / "config.json").write_text(json.dumps(CFG.to_json(), indent=2))

banner("WORKSPACE READY", f"{CFG.root}  |  {_free_root:.1f} GB free")
for d in sorted(p.name for p in CFG.root.iterdir() if p.is_dir()):
    CONSOLE.print(f"   [cyan]•[/cyan] {d}/")
log_ok("Folder contract matches the project specification.")

In [ ]:
# =====================================================================
#  CELL 5 | CANONICAL TAXONOMY
#  One authoritative class list shared by: YOLO training, products.csv,
#  and the NLP module. Also the merge map that collapses synonymous
#  class names coming from heterogeneous public datasets.
# =====================================================================
import re
from collections import Counter, defaultdict

# Ordered, stable taxonomy. Index == YOLO class id. NEVER reorder after
# training: label files store integer ids.
CANONICAL_CLASSES: List[str] = [
    "Potato Chips", "Corn Chips", "Snack", "Chocolate", "Candy", "Biscuit",
    "Cookie", "Rice", "Pasta", "Sugar", "Salt", "Flour", "Coffee", "Tea",
    "Milk", "Juice", "Yogurt", "Cheese", "Butter", "Soft Drink", "Water",
    "Energy Drink", "Frozen Food", "Beans", "Canned Food", "Sauce", "Ketchup",
    "Mayonnaise", "Cooking Oil", "Spice", "Instant Noodles", "Bread",
    "Cereal", "Honey", "Jam", "Soap", "Shampoo", "Toothpaste",
    "Cleaning Product", "Laundry Product", "Pet Food", "Baby Product",
    "Beverage", "Packaged Product",
]
CLASS_TO_ID = {name: i for i, name in enumerate(CANONICAL_CLASSES)}

# High-level grouping used for reporting and for the NLP `category` column.
CATEGORY_OF = {
    **{c: "Snacks" for c in ["Potato Chips", "Corn Chips", "Snack", "Biscuit",
                             "Cookie", "Chocolate", "Candy"]},
    **{c: "Beverages" for c in ["Coffee", "Tea", "Juice", "Soft Drink",
                                "Water", "Energy Drink", "Beverage"]},
    **{c: "Dairy" for c in ["Milk", "Yogurt", "Cheese", "Butter"]},
    **{c: "Pantry" for c in ["Rice", "Pasta", "Sugar", "Salt", "Flour",
                             "Beans", "Canned Food", "Sauce", "Ketchup",
                             "Mayonnaise", "Cooking Oil", "Spice",
                             "Instant Noodles", "Bread", "Cereal", "Honey",
                             "Jam", "Frozen Food"]},
    **{c: "Personal Care" for c in ["Soap", "Shampoo", "Toothpaste"]},
    **{c: "Household" for c in ["Cleaning Product", "Laundry Product"]},
    "Pet Food": "Pet", "Baby Product": "Baby",
    "Packaged Product": "General",
}

# Synonym → canonical. Keys are normalised (lowercase, no punctuation).
_SYNONYMS: Dict[str, str] = {
    # chips family
    "chips": "Potato Chips", "crisps": "Potato Chips",
    "potato chip": "Potato Chips", "snack chips": "Potato Chips",
    "french fries": "Potato Chips", "tortilla chips": "Corn Chips",
    "nachos": "Corn Chips", "doritos": "Corn Chips", "popcorn": "Snack",
    "pretzel": "Snack", "snacks": "Snack", "crackers": "Biscuit",
    "cracker": "Biscuit", "biscuits": "Biscuit", "wafer": "Biscuit",
    "cookies": "Cookie", "chocolate bar": "Chocolate", "cocoa": "Chocolate",
    "sweets": "Candy", "gum": "Candy", "lollipop": "Candy",
    # beverages
    "coke": "Soft Drink", "coca cola": "Soft Drink", "cocacola": "Soft Drink",
    "pepsi": "Soft Drink", "soda": "Soft Drink", "cola": "Soft Drink",
    "carbonated drink": "Soft Drink", "soft drinks": "Soft Drink",
    "drink": "Beverage", "beverages": "Beverage", "bottle": "Beverage",
    "bottled water": "Water", "mineral water": "Water",
    "orange juice": "Juice", "fruit juice": "Juice",
    "coffee cup": "Coffee", "instant coffee": "Coffee", "espresso": "Coffee",
    "tea bag": "Tea", "green tea": "Tea", "energy drinks": "Energy Drink",
    # dairy
    "dairy product": "Milk", "dairy": "Milk", "milk carton": "Milk",
    "yoghurt": "Yogurt", "yogurt cup": "Yogurt", "margarine": "Butter",
    # pantry
    "noodle": "Instant Noodles", "noodles": "Instant Noodles",
    "ramen": "Instant Noodles", "spaghetti": "Pasta", "macaroni": "Pasta",
    "tin can": "Canned Food", "canned goods": "Canned Food",
    "tinned food": "Canned Food", "olive oil": "Cooking Oil",
    "sunflower oil": "Cooking Oil", "oil": "Cooking Oil",
    "tomato sauce": "Sauce", "condiment": "Sauce", "mustard": "Sauce",
    "spices": "Spice", "seasoning": "Spice", "breakfast cereal": "Cereal",
    "cornflakes": "Cereal", "jelly": "Jam", "marmalade": "Jam",
    "frozen vegetables": "Frozen Food", "frozen vegetable": "Frozen Food",
    "bread loaf": "Bread", "bagel": "Bread", "bun": "Bread",
    "legume": "Beans", "lentils": "Beans", "chickpeas": "Beans",
    # care / household
    "hand soap": "Soap", "bar soap": "Soap", "body wash": "Soap",
    "hair care": "Shampoo", "conditioner": "Shampoo",
    "tooth paste": "Toothpaste", "dental care": "Toothpaste",
    "detergent": "Laundry Product", "laundry detergent": "Laundry Product",
    "fabric softener": "Laundry Product", "cleaner": "Cleaning Product",
    "cleaning supplies": "Cleaning Product", "bleach": "Cleaning Product",
    "diaper": "Baby Product", "baby food": "Baby Product",
    "dog food": "Pet Food", "cat food": "Pet Food",
    # generic fallbacks from broad datasets
    "food": "Packaged Product", "packaged goods": "Packaged Product",
    "product": "Packaged Product", "box": "Packaged Product",
    "tin": "Canned Food", "jar": "Packaged Product",
}

CLASS_MERGE_LOG: List[Tuple[str, str]] = []

def _normalise_token(raw: str) -> str:
    token = re.sub(r"[_\\-]+", " ", str(raw)).strip().lower()
    token = re.sub(r"[^a-z0-9 ]", "", token)
    return re.sub(r"\\s+", " ", token).strip()

def canonicalise(raw_name: str) -> Optional[str]:
    """Map any incoming class name to the canonical taxonomy, or None.

    Resolution order: exact canonical → explicit synonym → singular/plural →
    substring containment. Returns None when nothing matches so unknown
    classes are dropped rather than silently mislabelled.
    """
    token = _normalise_token(raw_name)
    if not token:
        return None

    exact = {_normalise_token(c): c for c in CANONICAL_CLASSES}
    if token in exact:
        return exact[token]
    if token in _SYNONYMS:
        resolved = _SYNONYMS[token]
        CLASS_MERGE_LOG.append((raw_name, resolved))
        return resolved

    singular = token[:-1] if token.endswith("s") else token + "s"
    if singular in exact:
        return exact[singular]
    if singular in _SYNONYMS:
        resolved = _SYNONYMS[singular]
        CLASS_MERGE_LOG.append((raw_name, resolved))
        return resolved

    for candidate_norm, candidate in exact.items():
        if candidate_norm in token or token in candidate_norm:
            CLASS_MERGE_LOG.append((raw_name, candidate))
            return candidate
    for syn, canonical in _SYNONYMS.items():
        if syn in token:
            CLASS_MERGE_LOG.append((raw_name, canonical))
            return canonical
    return None

# Self-test: a silent taxonomy bug destroys the whole pipeline, so assert.
_TAXONOMY_TESTS = {
    "Coca-Cola": "Soft Drink", "coke": "Soft Drink", "CRISPS": "Potato Chips",
    "Dairy Product": "Milk", "Tin can": "Canned Food", "Juice": "Juice",
    "laundry_detergent": "Laundry Product", "quantum physics": None,
}
for probe, expected in _TAXONOMY_TESTS.items():
    got = canonicalise(probe)
    assert got == expected, f"taxonomy regression: {probe} -> {got}"

CONSOLE.print(Panel(
    f"[bold]{len(CANONICAL_CLASSES)}[/bold] canonical classes across "
    f"[bold]{len(set(CATEGORY_OF.values()))}[/bold] categories\n"
    f"[bold]{len(_SYNONYMS)}[/bold] synonym rules · self-test passed",
    title="CANONICAL RETAIL TAXONOMY", border_style="green"))

╭──────────────────────────────────────── CANONICAL RETAIL TAXONOMY ─────────────────────────────────────────╮
│ 44 canonical classes across 9 categories                                                                   │
│ 99 synonym rules · self-test passed                                                                        │
╰────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
# =====================================================================
# CELL 6 | DOWNLOAD ROBOFLOW DATASETS & SAVE TO GOOGLE DRIVE SAFELY
# =====================================================================
!pip install -q roboflow rich
import os
import shutil
import yaml
from pathlib import Path
from rich.console import Console
from roboflow import Roboflow

console = Console()

# 1. إعداد المسار الأساسي على الدرايف (عشان الداتا متضيعش تاني أبداً)
DRIVE_BASE_DIR = Path("/content/drive/MyDrive/NTI tasks/AI Shopping Assistant/Roboflow_Data")
DRIVE_BASE_DIR.mkdir(parents=True, exist_ok=True)

rf = Roboflow(api_key="72Ra0VqyC5nnTE1mWJAV")

# 2. قائمة المشاريع بتاعتك (تم تجهيزها من الأكواد اللي بعتها)
datasets_to_download = [
    ("omerfarukakgul1999-gmail-com", "shopeye", 4),
    ("aleynas-workspace-enysf", "market-urunleri-muo18", 5),
    ("my-space-lw93l", "products-detection-00asl", 27),
    ("custom-yolov7", "detection-xcpss", 15)
]

console.print(f"[bold cyan]🚀 جاري تحميل البيانات وحفظها مباشرة في الدرايف...[/bold cyan]")
console.print(f"[bold cyan]📁 مسار الحفظ الدائم: {DRIVE_BASE_DIR}[/bold cyan]")

for workspace, project, version in datasets_to_download:
    console.print(f"\n[bold yellow]📥 جاري معالجة مشروع: {project} (v{version})[/bold yellow]")

    # مسار الحفظ النهائي للمشروع ده جوه الدرايف
    project_drive_path = DRIVE_BASE_DIR / f"{project}_v{version}"

    if project_drive_path.exists():
        console.print(f"[bold green]✅ المشروع موجود بالفعل على الدرايف (تم تخطي التحميل).[/bold green]")
        continue

    try:
        # التحميل لمساحة الكولاب المؤقتة الأول (لأن الدرايف بطيء في التحميل المباشر)
        proj = rf.workspace(workspace).project(project)
        dataset = proj.version(version).download("yolov11") # تم ضبط الفورمات على yolov11
        temp_path = Path(dataset.location)

        # النقل الآمن للدرايف
        console.print(f"🚚 جاري النقل الآمن لـ Google Drive...")
        shutil.copytree(temp_path, project_drive_path, dirs_exist_ok=True)

        # 🔥 الأمان من الأخطاء: تعديل ملف data.yaml ليتوافق مع مسار الدرايف الجديد 🔥
        yaml_path = project_drive_path / "data.yaml"
        if yaml_path.exists():
            with open(yaml_path, 'r', encoding='utf-8') as f:
                yaml_data = yaml.safe_load(f)

            # تحديث المسارات لتكون صحيحة 100% لـ YOLO عشان نتجنب إيرور التدريب
            yaml_data['path'] = str(project_drive_path)
            yaml_data['train'] = "train/images"
            yaml_data['val'] = "valid/images"
            if 'test' in yaml_data:
                yaml_data['test'] = "test/images"

            with open(yaml_path, 'w', encoding='utf-8') as f:
                yaml.dump(yaml_data, f, sort_keys=False)

            console.print(f"[bold green]🔧 تم تصحيح مسارات data.yaml بنجاح لتجنب أي إيرور.[/bold green]")

        console.print(f"[bold blue]🎉 تم الحفظ بنجاح وتأمين البيانات.[/bold blue]")

    except Exception as e:
        console.print(f"[bold red]❌ خطأ في تحميل {project}: {e}[/bold red]")

console.print(f"\n[bold magenta]💡 ملاحظة: الووكرز 32 ثابتة معانا في سيل التدريب زي ما اتفقنا![/bold magenta]")

🚀 جاري تحميل البيانات وحفظها مباشرة في الدرايف...

📁 مسار الحفظ الدائم: /content/drive/MyDrive/NTI tasks/AI Shopping Assistant/Roboflow_Data

📥 جاري معالجة مشروع: shopeye (v4)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to shopeye-4 in yolov11:: 100%|██████████| 2768/2768 [00:00<00:00, 8855.90it/s]


...


Extracting Dataset Version Zip to Detection-15 in yolov11:: 100%|██████████| 2309/2309 [00:00<00:00, 3421.76it/s]


🚚 جاري النقل الآمن لـ Google Drive...

🔧 تم تصحيح مسارات data.yaml بنجاح لتجنب أي إيرور.

🎉 تم الحفظ بنجاح وتأمين البيانات.

💡 ملاحظة: الووكرز 32 ثابتة معانا في سيل التدريب زي ما اتفقنا!

In [ ]:
import os
from pathlib import Path

drive_path = Path("/content/drive/MyDrive/NTI tasks/AI Shopping Assistant/Roboflow_Data")

if drive_path.exists():
    folders = [f.name for f in drive_path.iterdir() if f.is_dir()]
    print("📁 الفولدرات الموجودة على الدرايف حالياً:")
    for folder in folders:
        print(f" - {folder}")
else:
    print("❌ المسار غير موجود، تأكد من عمل Drive Mount.")

📁 الفولدرات الموجودة على الدرايف حالياً:
 - shopeye_v4
 - market-urunleri-muo18_v5
 - products-detection-00asl_v27
 - detection-xcpss_v15


In [ ]:
!pip install "numpy<2.0.0" --force-reinstall

In [ ]:
# =====================================================================
#  CELL 6 | DATASET ACQUISITION (FULL DATASET - STABLE VERSION)
# =====================================================================
import json
import os
import time
import shutil
from pathlib import Path
from typing import Any, Dict, List, Optional
from rich.console import Console
from rich.table import Table

CONSOLE = Console(width=110)
ACQUISITION_LOG: List[Dict[str, Any]] = []
RAW_SOURCES: List[Dict[str, Any]] = []

_OI_CANDIDATES = [
    "Snack", "Juice", "Coffee", "Tea", "Milk", "Dairy Product", "Cheese",
    "Butter", "Bread", "Cookie", "Candy", "Chocolate", "Bottle",
    "Tin can", "Drink", "Soft drink", "Beer", "Wine", "Coffee cup",
    "French fries", "Popcorn", "Pretzel", "Cake", "Doughnut", "Honeycomb",
    "Common fig", "Condiment", "Ketchup", "Mustard", "Salt and pepper shakers",
    "Food", "Fast food", "Packaged goods", "Personal care", "Cosmetics",
    "Toothbrush", "Facial tissue holder", "Shampoo", "Soap dispenser",
    "Dog food", "Cat food", "Baby bottle", "Vegetable", "Fruit",
]

def _resolve_open_images_classes() -> List[str]:
    try:
        from fiftyone.utils.openimages import get_classes
        available = set(get_classes(version="v7"))
    except Exception:
        available = {"Snack", "Juice", "Coffee", "Milk", "Bottle", "Tin can", "Cookie", "Candy", "Bread", "Food"}
    resolved = [c for c in _OI_CANDIDATES if c in available and canonicalise(c) is not None]
    return resolved

def acquire_open_images() -> Optional[Dict[str, Any]]:
    marker = CFG.raw_dir / "open_images.done"
    export_dir = CFG.raw_dir / "open_images_yolo"

    if marker.exists() and (export_dir / "labels").exists():
        log_ok("Open Images subset already present (cached).")
        return {"name": "open_images_v7", "path": export_dir, "format": "yolo"}

    try:
        import fiftyone as fo
        import fiftyone.zoo as foz
    except Exception as exc:
        log_err(f"FiftyOne import error: {exc}")
        return None

    oi_classes = _resolve_open_images_classes()
    if not oi_classes:
        return None

    target_samples = max(1000, CFG.target_images_per_class * len(oi_classes))
    log_step(f"Downloading FULL Open Images V7 for {len(oi_classes)} labels (Target ≈ {target_samples} images)...")

    dataset = None
    for attempt in range(1, 4):
        try:
            if "oi_grocery" in fo.list_datasets():
                fo.delete_dataset("oi_grocery")
            dataset = foz.load_zoo_dataset(
                "open-images-v7", split="train", label_types=["detections"],
                classes=oi_classes, max_samples=target_samples, seed=CFG.seed, shuffle=True, dataset_name="oi_grocery"
            )
            break
        except Exception as exc:
            log_warn(f"Download attempt {attempt} failed: {exc}")
            time.sleep(5)

    if not dataset:
        return None

    export_dir.mkdir(parents=True, exist_ok=True)
    dataset.export(export_dir=str(export_dir), dataset_type=fo.types.YOLOv5Dataset, label_field="ground_truth", classes=oi_classes, split="train")

    marker.write_text(json.dumps({"samples": len(dataset)}))
    ACQUISITION_LOG.append({"source": "open-images-v7", "samples": len(dataset), "classes": len(oi_classes), "status": "ok"})
    return {"name": "open_images_v7", "path": export_dir, "format": "yolo"}

banner("STAGE 1 — DATASET ACQUISITION", "FULL DATASET VERSION (STABLE)")

source = acquire_open_images()
if source:
    RAW_SOURCES.append(source)

if not RAW_SOURCES:
    raise RuntimeError("CRITICAL: Dataset acquisition failed.")

_acq_table = Table(title="ACQUISITION SUMMARY", header_style="bold cyan")
for col in ("Source", "Samples", "Classes", "Status"):
    _acq_table.add_column(col)
for row in ACQUISITION_LOG:
    _acq_table.add_row(row["source"], str(row["samples"]), str(row["classes"]), row["status"])
CONSOLE.print(_acq_table)

In [ ]:
# =====================================================================
# CELL 6.2 | DOWNLOAD ADDITIONAL DATASETS & SAVE TO DRIVE SAFELY
# =====================================================================
!pip install -q roboflow rich
import os
import shutil
import yaml
from pathlib import Path
from rich.console import Console
from roboflow import Roboflow

console = Console()

# نفس مسار الحفظ الدائم بتاعنا
DRIVE_BASE_DIR = Path("/content/drive/MyDrive/NTI tasks/AI Shopping Assistant/Roboflow_Data")
DRIVE_BASE_DIR.mkdir(parents=True, exist_ok=True)

rf = Roboflow(api_key="72Ra0VqyC5nnTE1mWJAV")

# قائمة المشاريع الجديدة اللي أنت ضفتها
new_datasets_to_download = [
    ("virtual7069", "virtualcart", 11),
    ("object-recognition-naoui", "nutrition-qz6dz", 3),
    ("easy-cart", "super-market-qszif", 2),
    ("smartcart-orubg", "mini-market-items", 5)
]

console.print(f"[bold cyan]🚀 جاري تحميل حزمة البيانات الإضافية وحفظها في الدرايف...[/bold cyan]")

for workspace, project, version in new_datasets_to_download:
    console.print(f"\n[bold yellow]📥 جاري معالجة مشروع: {project} (v{version})[/bold yellow]")

    project_drive_path = DRIVE_BASE_DIR / f"{project}_v{version}"

    if project_drive_path.exists():
        console.print(f"[bold green]✅ المشروع ده نزل وموجود فعلاً على الدرايف.[/bold green]")
        continue

    try:
        # التحميل بصيغة yolov11
        proj = rf.workspace(workspace).project(project)
        dataset = proj.version(version).download("yolov11")
        temp_path = Path(dataset.location)

        # النقل للدرايف
        console.print(f"🚚 جاري النقل الآمن لـ Google Drive...")
        shutil.copytree(temp_path, project_drive_path, dirs_exist_ok=True)

        # تأمين مسارات data.yaml
        yaml_path = project_drive_path / "data.yaml"
        if yaml_path.exists():
            with open(yaml_path, 'r', encoding='utf-8') as f:
                yaml_data = yaml.safe_load(f)

            yaml_data['path'] = str(project_drive_path)
            yaml_data['train'] = "train/images"
            yaml_data['val'] = "valid/images"
            if 'test' in yaml_data:
                yaml_data['test'] = "test/images"

            with open(yaml_path, 'w', encoding='utf-8') as f:
                yaml.dump(yaml_data, f, sort_keys=False)

            console.print(f"[bold green]🔧 تم تصحيح مسارات data.yaml بنجاح.[/bold green]")

        console.print(f"[bold blue]🎉 تم الحفظ بنجاح وتأمين البيانات.[/bold blue]")

    except Exception as e:
        console.print(f"[bold red]❌ خطأ في تحميل {project}: {e}[/bold red]")

console.print(f"\n[bold magenta]🔥 الداتا الإضافية بقت جاهزة معانا في الدرايف![/bold magenta]")

🚀 جاري تحميل حزمة البيانات الإضافية وحفظها في الدرايف...

📥 جاري معالجة مشروع: virtualcart (v11)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to virtualcart-11 in yolov11:: 100%|██████████| 2307/2307 [00:00<00:00, 5232.22it/s]


🚚 جاري النقل الآمن لـ Google Drive...

...


Extracting Dataset Version Zip to mini-market-items-5 in yolov11:: 100%|██████████| 763/763 [00:00<00:00, 7063.42it/s]


🚚 جاري النقل الآمن لـ Google Drive...

🔧 تم تصحيح مسارات data.yaml بنجاح.

🎉 تم الحفظ بنجاح وتأمين البيانات.

🔥 الداتا الإضافية بقت جاهزة معانا في الدرايف!

In [ ]:
# =====================================================================
# DEEP DATASET EXPLORER | ADVANCED STATS & IMBALANCE CHECK
# =====================================================================
import os
import yaml
from pathlib import Path
from collections import Counter
from rich.console import Console
from rich.table import Table

console = Console()
DRIVE_BASE_DIR = Path("/content/drive/MyDrive/NTI tasks/AI Shopping Assistant/Roboflow_Data")

if not DRIVE_BASE_DIR.exists():
    console.print(f"[bold red]❌ مسار الداتا غير موجود: {DRIVE_BASE_DIR}[/bold red]")
else:
    console.print(f"[bold cyan]🔍 جاري الفحص العميق للمشاريع في المسار: {DRIVE_BASE_DIR}[/bold cyan]\n")

    total_all_projects_instances = Counter()
    total_global_instances = 0

    for project_dir in DRIVE_BASE_DIR.iterdir():
        if not project_dir.is_dir(): continue

        yaml_path = project_dir / "data.yaml"
        if not yaml_path.exists(): continue

        try:
            with open(yaml_path, 'r', encoding='utf-8') as f:
                yaml_data = yaml.safe_load(f)

            classes = yaml_data.get('names', [])
            if isinstance(classes, dict):
                classes = [classes[i] for i in sorted(classes.keys())]

            # إحصائيات التقسيمات والصور الفارغة
            split_counts = {'train': 0, 'valid': 0, 'test': 0}
            empty_images_count = 0

            class_instances = Counter()
            images_with_class = Counter()

            # فحص كل الفولدرات (Train, Valid, Test)
            for split in ['train', 'valid', 'test']:
                labels_dir = project_dir / split / "labels"
                if labels_dir.exists():
                    for label_file in labels_dir.glob("*.txt"):
                        split_counts[split] += 1

                        # التحقق من الصور الفارغة (Background)
                        if os.path.getsize(label_file) == 0:
                            empty_images_count += 1
                            continue

                        found_classes = set()
                        is_empty = True

                        with open(label_file, 'r', encoding='utf-8') as lf:
                            for line in lf:
                                parts = line.strip().split()
                                if not parts: continue
                                class_id = int(parts[0])
                                class_instances[class_id] += 1
                                found_classes.add(class_id)
                                is_empty = False

                        if is_empty:
                            empty_images_count += 1

                        for cid in found_classes:
                            images_with_class[cid] += 1

            # 1. طباعة معلومات المشروع الأساسية
            console.print(f"[bold yellow]📁 مشروع: {project_dir.name}[/bold yellow]")
            console.print(f"   [dim]التقسيم: التدريب ({split_counts['train']}) | التحقق ({split_counts['valid']}) | الاختبار ({split_counts['test']})[/dim]")
            if empty_images_count > 0:
                console.print(f"   [dim]صور الخلفية (بدون منتجات): {empty_images_count} صورة[/dim]")

            # 2. طباعة جدول الكلاسات العميق
            table = Table(show_header=True, header_style="bold magenta")
            table.add_column("ID", justify="center", style="dim")
            table.add_column("اسم الكلاس (Class)", style="cyan")
            table.add_column("الصور", justify="right", style="blue")
            table.add_column("المنتجات", justify="right", style="green")
            table.add_column("الكثافة (Avg/Img)", justify="right", style="yellow")

            for class_id, class_name in enumerate(classes):
                instances = class_instances.get(class_id, 0)
                images = images_with_class.get(class_id, 0)
                density = round(instances / images, 1) if images > 0 else 0

                table.add_row(str(class_id), str(class_name), str(images), str(instances), f"{density} x")
                total_all_projects_instances[class_name] += instances
                total_global_instances += instances

            console.print(table)
            console.print("-" * 60)

        except Exception as e:
            console.print(f"[bold red]❌ خطأ في قراءة مشروع {project_dir.name}: {e}[/bold red]")

    # 3. الملخص العالمي (مهم جداً لاكتشاف الـ Imbalance)
    console.print("\n[bold green]🏆 الملخص الشامل: تحليل التوازن (Class Imbalance)[/bold green]")
    summary_table = Table(show_header=True, header_style="bold blue")
    summary_table.add_column("اسم الكلاس", style="cyan")
    summary_table.add_column("إجمالي المنتجات", justify="right", style="green")
    summary_table.add_column("النسبة من الداتا (%)", justify="right", style="magenta")

    for cls_name, count in total_all_projects_instances.most_common():
        percentage = round((count / total_global_instances) * 100, 2) if total_global_instances > 0 else 0
        # تمييز الكلاسات الضعيفة باللون الأحمر
        pct_str = f"[red]{percentage}%[/red]" if percentage < 1.0 else f"{percentage}%"
        summary_table.add_row(str(cls_name), str(count), pct_str)

    console.print(summary_table)
    console.print(f"\n[bold white]إجمالي المنتجات (Bounding Boxes) في كل المشاريع: {total_global_instances}[/bold white]")

🔍 جاري الفحص العميق للمشاريع في المسار: /content/drive/MyDrive/NTI tasks/AI Shopping Assistant/Roboflow_Data

KeyboardInterrupt: 

In [ ]:
# =====================================================================
# CELL 7 | SMART MERGER & DATA CLEANUP (32 WORKERS)
# =====================================================================
import os
import shutil
import yaml
import random
from pathlib import Path
from tqdm import tqdm
from rich.table import Table
from rich.console import Console
from collections import Counter
from concurrent.futures import ProcessPoolExecutor, as_completed

console = Console()

FAST_DATASET_DIR = Path("/content/yolo_dataset")
if FAST_DATASET_DIR.exists():
    shutil.rmtree(FAST_DATASET_DIR, ignore_errors=True)

for split in ['train', 'valid']:
    (FAST_DATASET_DIR / split / 'images').mkdir(parents=True, exist_ok=True)
    (FAST_DATASET_DIR / split / 'labels').mkdir(parents=True, exist_ok=True)

DRIVE_BASE_DIR = Path("/content/drive/MyDrive/NTI tasks/AI Shopping Assistant/Roboflow_Data")

# 1. القائمة البيضاء (الخريطة الذكية) - أي كلاس مش هنا هيتم حذفه
CLASS_MAPPING = {
    'Detergent': ['Sivi_deterjan', 'Toz_deterjan', 'Temizlik_Deterjan', 'Bulasik_makinesi_deterjani'],
    'Cooking_Oil': ['Sivi_yag', 'Zeytinyagi'],
    'Pasta_Noodles': ['Makarna', 'indomie green', 'indomie yellow'],
    'Tea_Coffee': ['Cay', 'Lipton Early Grey', 'Lipton Ice Tea', 'Nescafe Gold'],
    'Soda': ['Kola', 'Coca Cola 1 Liter', 'Coca Cola 330', 'kinza cola', 'kinza lemon', 'kinza saudi cocktail', 'kinza soda water', 'kinza zero'],
    'Milk': ['Sut', 'Milk', 'Milk-Large', 'Milk-Small', 'Almarai-Milk-HalfCream-Large', 'Lactel-Milk-FullCream-Large'],
    'Cheese': ['Peynir', 'President Cheese', 'cheese', 'kiri cheese'],
    'Juice': ['Juice-Large', 'Juice-Small', 'juice', 'Chocolate juice', 'Juhayna-Juice-Apple-Small', 'Beyti-Juice-Large', 'sun top mixed'],
    'Chocolate_Candy': ['Chocolate', 'chocolate', 'Nutella', 'Browni Intense', 'm and m Chocolate', 'Nestle Bitter', 'Galaxy-Chocolate-Jewels', 'Kitkat-Chocolate-White', 'Snickers-Chocolate', 'Kitkat-Chocolate-Chunky-Original', 'Milka-Chocolate-OreoChoco', 'Mentos', 'Vivident Fruit Swing', 'bounty chocolate', 'kinder', 'kitkat', 'mars chocolate', 'snickers', 'twix chocolate'],
    'Chips': ['Pringles Paprika', 'Pringles Sour Cream', 'Doritos Taco', 'Ruffles Patato', 'tasali chips'],
    'Tomato_Paste': ['Salca'],
    'Yogurt': ['Yogurt', 'yogurt', 'activia yogurt'],
    'Cereal_Biscuits': ['Cereal', 'Nestle-Cereal-Fitness-Original-Large', 'Nestle-Cereal-Lion-Caramel-Chocolate-Large', 'Nestle-Cereal-Nesquik-CocoCrush-Large', 'Biscuits'],
    'Eggs': ['Yumurta'],
    'Water': ['water'],
    'Sauces_Jam': ['mayonnaise', 'Jam']
}

REVERSE_MAP = {raw: final for final, raw_list in CLASS_MAPPING.items() for raw in raw_list}
FINAL_CLASSES = list(CLASS_MAPPING.keys())
CLASS_TO_ID = {name: idx for idx, name in enumerate(FINAL_CLASSES)}

# 2. المعالجة المتوازية
def process_single_image(args):
    img_path, lbl_path, raw_classes = args
    local_stats = Counter()
    split = 'train' if random.random() < 0.85 else 'valid' # 85% تدريب

    valid_boxes = []
    try:
        with open(lbl_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()

        for line in lines:
            parts = line.strip().split()
            if len(parts) < 5: continue

            raw_id = int(parts[0])
            cx, cy, w, h = map(float, parts[1:5])

            if 0 <= raw_id < len(raw_classes):
                raw_name = raw_classes[raw_id]
                final_name = REVERSE_MAP.get(raw_name) # لو مش في الخريطة، هيرجع None ويتجاهله

                if final_name:
                    final_id = CLASS_TO_ID[final_name]
                    valid_boxes.append(f"{final_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n")
                    local_stats[final_name] += 1
    except Exception:
        pass

    # إذا الصورة مفهاش بوكسات صالحة (كل الكلاسات زبالة)، هتبقي Background Image مفيدة
    dst_img = FAST_DATASET_DIR / split / 'images' / f"{img_path.parent.parent.parent.name}_{img_path.name}"
    dst_lbl = FAST_DATASET_DIR / split / 'labels' / f"{img_path.parent.parent.parent.name}_{img_path.stem}.txt"

    shutil.copyfile(img_path, dst_img)
    with open(dst_lbl, 'w', encoding='utf-8') as f:
        f.writelines(valid_boxes)

    if valid_boxes:
        local_stats["images_with_objects"] += 1
    else:
        local_stats["background_images"] += 1

    return True, local_stats

console.print("[bold cyan]🔍 جاري تجميع الصور واستبعاد الكلاسات غير المعروفة...[/bold cyan]")
tasks = []

for project_dir in DRIVE_BASE_DIR.iterdir():
    if not project_dir.is_dir(): continue
    yaml_path = project_dir / "data.yaml"
    if not yaml_path.exists(): continue

    with open(yaml_path, 'r') as f:
        y_data = yaml.safe_load(f)
        raw_classes = y_data.get('names', [])
        if isinstance(raw_classes, dict):
            raw_classes = [raw_classes[i] for i in sorted(raw_classes.keys())]

    for split in ['train', 'valid', 'test']:
        img_dir = project_dir / split / 'images'
        lbl_dir = project_dir / split / 'labels'

        if img_dir.exists() and lbl_dir.exists():
            for img_path in img_dir.glob("*.*"):
                if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                    lbl_path = lbl_dir / f"{img_path.stem}.txt"
                    if lbl_path.exists():
                        tasks.append((img_path, lbl_path, raw_classes))

WORKERS = 32
console.print(f"🚀 بدء معالجة {len(tasks)} صورة باستخدام {WORKERS} Workers...")

FINAL_STATS = Counter()
successful_images = 0

with ProcessPoolExecutor(max_workers=WORKERS) as executor:
    futures = {executor.submit(process_single_image, task): task for task in tasks}
    for future in tqdm(as_completed(futures), total=len(tasks), desc="Cleaning & Merging"):
        success, stats = future.result()
        FINAL_STATS.update(stats)
        if success: successful_images += 1

final_yaml_path = FAST_DATASET_DIR / "data.yaml"
with open(final_yaml_path, 'w', encoding='utf-8') as f:
    f.write(f"path: {FAST_DATASET_DIR}\n")
    f.write("train: train/images\n")
    f.write("val: valid/images\n\n")
    f.write("names:\n")
    for idx, name in enumerate(FINAL_CLASSES):
        f.write(f"  {idx}: {name}\n")

console.print(f"\n[bold green]✅ تم التنظيف والدمج بنجاح![/bold green]")
console.print(f"📸 إجمالي الصور المنقولة: {successful_images} (بها منتجات: {FINAL_STATS['images_with_objects']} | صور خلفية: {FINAL_STATS['background_images']})")

table = Table(title="🏆 إحصائيات الكلاسات الـ 16 النظيفة الجاهزة للتدريب", header_style="bold magenta")
table.add_column("ID", style="dim")
table.add_column("اسم الكلاس (Class)", style="cyan")
table.add_column("عدد المنتجات الفعلي", justify="right", style="green")

for idx, name in enumerate(FINAL_CLASSES):
    table.add_row(str(idx), name, str(FINAL_STATS[name]))

console.print(table)

🔍 جاري تجميع الصور واستبعاد الكلاسات غير المعروفة...

🚀 بدء معالجة 8778 صورة باستخدام 32 Workers...

Cleaning & Merging: 100%|██████████| 8778/8778 [05:43<00:00, 25.54it/s]


✅ تم التنظيف والدمج بنجاح!

📸 إجمالي الصور المنقولة: 8778 (بها منتجات: 6955 | صور خلفية: 1823)

   🏆 إحصائيات الكلاسات الـ 16 النظيفة الجاهزة   
                     للتدريب                     
┏━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃ ID ┃ اسم الكلاس (Class) ┃ عدد المنتجات الفعلي ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ 0  │ Detergent          │                8424 │
│ 1  │ Cooking_Oil        │                5713 │
│ 2  │ Pasta_Noodles      │                2679 │
│ 3  │ Tea_Coffee         │                2670 │
│ 4  │ Soda               │                2668 │
│ 5  │ Milk               │                2541 │
│ 6  │ Cheese             │                2561 │
│ 7  │ Juice              │                 555 │
│ 8  │ Chocolate_Candy    │                1782 │
│ 9  │ Chips              │                 727 │
│ 10 │ Tomato_Paste       │                3123 │
│ 11 │ Yogurt             │                1406 │
│ 12 │ Cereal_Biscuits    │                 472 │
│ 13 │ Eggs               │                 249 │
│ 14 │ Water              │                 149 │
│ 15 │ Sauces_Jam         │                 362 │
└────┴────────────────────┴─────────────────────┘

In [ ]:
# =====================================================================
# CELL 7 | SMART DATA CLEANING & STAGING (32 WORKERS)
# =====================================================================
import os
import shutil
from pathlib import Path
from tqdm import tqdm
from rich.table import Table
from rich.console import Console
from collections import Counter
from concurrent.futures import ProcessPoolExecutor, as_completed
import yaml

console = Console()

FAST_STAGING_DIR = Path("/content/temp_staging")
if FAST_STAGING_DIR.exists():
    shutil.rmtree(FAST_STAGING_DIR, ignore_errors=True)

(FAST_STAGING_DIR / 'images').mkdir(parents=True, exist_ok=True)
(FAST_STAGING_DIR / 'labels').mkdir(parents=True, exist_ok=True)

DRIVE_BASE_DIR = Path("/content/drive/MyDrive/NTI tasks/AI Shopping Assistant/Roboflow_Data")

# 1. القائمة البيضاء (الخريطة الذكية للـ 16 كلاس)
CLASS_MAPPING = {
    'Detergent': ['Sivi_deterjan', 'Toz_deterjan', 'Temizlik_Deterjan', 'Bulasik_makinesi_deterjani'],
    'Cooking_Oil': ['Sivi_yag', 'Zeytinyagi'],
    'Pasta_Noodles': ['Makarna', 'indomie green', 'indomie yellow'],
    'Tea_Coffee': ['Cay', 'Lipton Early Grey', 'Lipton Ice Tea', 'Nescafe Gold'],
    'Soda': ['Kola', 'Coca Cola 1 Liter', 'Coca Cola 330', 'kinza cola', 'kinza lemon', 'kinza saudi cocktail', 'kinza soda water', 'kinza zero'],
    'Milk': ['Sut', 'Milk', 'Milk-Large', 'Milk-Small', 'Almarai-Milk-HalfCream-Large', 'Lactel-Milk-FullCream-Large'],
    'Cheese': ['Peynir', 'President Cheese', 'cheese', 'kiri cheese'],
    'Juice': ['Juice-Large', 'Juice-Small', 'juice', 'Chocolate juice', 'Juhayna-Juice-Apple-Small', 'Beyti-Juice-Large', 'sun top mixed'],
    'Chocolate_Candy': ['Chocolate', 'chocolate', 'Nutella', 'Browni Intense', 'm and m Chocolate', 'Nestle Bitter', 'Galaxy-Chocolate-Jewels', 'Kitkat-Chocolate-White', 'Snickers-Chocolate', 'Kitkat-Chocolate-Chunky-Original', 'Milka-Chocolate-OreoChoco', 'Mentos', 'Vivident Fruit Swing', 'bounty chocolate', 'kinder', 'kitkat', 'mars chocolate', 'snickers', 'twix chocolate'],
    'Chips': ['Pringles Paprika', 'Pringles Sour Cream', 'Doritos Taco', 'Ruffles Patato', 'tasali chips'],
    'Tomato_Paste': ['Salca'],
    'Yogurt': ['Yogurt', 'yogurt', 'activia yogurt'],
    'Cereal_Biscuits': ['Cereal', 'Nestle-Cereal-Fitness-Original-Large', 'Nestle-Cereal-Lion-Caramel-Chocolate-Large', 'Nestle-Cereal-Nesquik-CocoCrush-Large', 'Biscuits'],
    'Eggs': ['Yumurta'],
    'Water': ['water'],
    'Sauces_Jam': ['mayonnaise', 'Jam']
}

REVERSE_MAP = {raw: final for final, raw_list in CLASS_MAPPING.items() for raw in raw_list}
FINAL_CLASSES = list(CLASS_MAPPING.keys())
CLASS_TO_ID = {name: idx for idx, name in enumerate(FINAL_CLASSES)}

def process_single_image(args):
    img_path, lbl_path, raw_classes = args
    local_stats = Counter()
    valid_boxes = []

    try:
        with open(lbl_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()

        for line in lines:
            parts = line.strip().split()
            if len(parts) < 5: continue

            raw_id = int(parts[0])
            cx, cy, w, h = map(float, parts[1:5])

            if 0 <= raw_id < len(raw_classes):
                raw_name = raw_classes[raw_id]
                final_name = REVERSE_MAP.get(raw_name)

                if final_name:
                    final_id = CLASS_TO_ID[final_name]
                    valid_boxes.append(f"{final_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n")
                    local_stats[final_name] += 1
    except Exception:
        pass

    dst_img = FAST_STAGING_DIR / 'images' / f"{img_path.parent.parent.parent.name}_{img_path.name}"
    dst_lbl = FAST_STAGING_DIR / 'labels' / f"{img_path.parent.parent.parent.name}_{img_path.stem}.txt"

    shutil.copyfile(img_path, dst_img)
    with open(dst_lbl, 'w', encoding='utf-8') as f:
        f.writelines(valid_boxes)

    return True, local_stats

console.print("[bold cyan]🔍 جاري تنظيف وتجميع الصور في مجلد الاستعداد (Staging)...[/bold cyan]")
tasks = []

for project_dir in DRIVE_BASE_DIR.iterdir():
    if not project_dir.is_dir(): continue
    yaml_path = project_dir / "data.yaml"
    if not yaml_path.exists(): continue

    with open(yaml_path, 'r') as f:
        y_data = yaml.safe_load(f)
        raw_classes = y_data.get('names', [])
        if isinstance(raw_classes, dict):
            raw_classes = [raw_classes[i] for i in sorted(raw_classes.keys())]

    for split in ['train', 'valid', 'test']:
        img_dir = project_dir / split / 'images'
        lbl_dir = project_dir / split / 'labels'

        if img_dir.exists() and lbl_dir.exists():
            for img_path in img_dir.glob("*.*"):
                if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                    lbl_path = lbl_dir / f"{img_path.stem}.txt"
                    if lbl_path.exists():
                        tasks.append((img_path, lbl_path, raw_classes))

WORKERS = 32
console.print(f"🚀 بدء معالجة {len(tasks)} صورة باستخدام {WORKERS} Workers...")

with ProcessPoolExecutor(max_workers=WORKERS) as executor:
    futures = {executor.submit(process_single_image, task): task for task in tasks}
    for future in tqdm(as_completed(futures), total=len(tasks), desc="Staging Data"):
        pass

console.print(f"\n[bold green]✅ تم تجميع الداتا النظيفة بنجاح في temp_staging![/bold green]")
console.print("[bold yellow]🔥 الآن شغل (Cell 8) لتقسيم الداتا...[/bold yellow]")

🔍 جاري تنظيف وتجميع الصور في مجلد الاستعداد (Staging)...

🚀 بدء معالجة 8778 صورة باستخدام 32 Workers...

Staging Data: 100%|██████████| 8778/8778 [01:23<00:00, 105.32it/s]


✅ تم تجميع الداتا النظيفة بنجاح في temp_staging!

🔥 الآن شغل (Cell 8) لتقسيم الداتا...

In [ ]:
# =====================================================================
#  CELL 8 | DATASET SPLIT & YOLO FORMATTING (32 WORKERS)
# =====================================================================
import os
import shutil
import random
import yaml
from pathlib import Path
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
from rich.table import Table
from rich.console import Console

CONSOLE = Console()

FAST_STAGING_DIR = Path("/content/temp_staging")
FINAL_DATASET_DIR = Path("/content/yolo_dataset")

# مسح الفولدر القديم لو موجود عشان نبدأ على نضافة
if FINAL_DATASET_DIR.exists():
    shutil.rmtree(FINAL_DATASET_DIR, ignore_errors=True)

# إنشاء المجلدات المطلوبة لـ YOLO (تدريب، تقييم، اختبار)
for split in ['train', 'val', 'test']:
    (FINAL_DATASET_DIR / split / 'images').mkdir(parents=True, exist_ok=True)
    (FINAL_DATASET_DIR / split / 'labels').mkdir(parents=True, exist_ok=True)

CONSOLE.print("[bold cyan]🚀 STAGE 3 — DATASET SPLITTING: Train / Val / Test[/bold cyan]")

staged_images_dir = FAST_STAGING_DIR / "images"
staged_labels_dir = FAST_STAGING_DIR / "labels"

# جمع كل الصور من الفولدر اللي اتعمل في Cell 7
valid_pairs = []
if staged_images_dir.exists():
    for img_path in staged_images_dir.glob("*.*"):
        lbl_path = staged_labels_dir / f"{img_path.stem}.txt"
        if lbl_path.exists():
            valid_pairs.append((img_path, lbl_path))

if not valid_pairs:
    CONSOLE.print("[bold red]❌ CRITICAL ERROR: 0 valid images found! تأكد من تشغيل Cell 7 أولاً.[/bold red]")
else:
    CONSOLE.print(f"[bold green]✅ تم العثور على {len(valid_pairs)} صورة نظيفة وجاهزة للتقسيم.[/bold green]")

    # خلط البيانات لضمان تنوع الصور في التدريب
    random.seed(42)
    random.shuffle(valid_pairs)

    # حساب النسب (80% تدريب - 10% تقييم - 10% اختبار)
    total = len(valid_pairs)
    train_end = int(total * 0.8)
    val_end = int(total * 0.9)

    train_pairs = valid_pairs[:train_end]
    val_pairs = valid_pairs[train_end:val_end]
    test_pairs = valid_pairs[val_end:]

    def prepare_tasks(pairs, split_name):
        tasks = []
        for img_path, lbl_path in pairs:
            dst_img = FINAL_DATASET_DIR / split_name / 'images' / img_path.name
            dst_lbl = FINAL_DATASET_DIR / split_name / 'labels' / lbl_path.name
            tasks.append((img_path, dst_img, lbl_path, dst_lbl))
        return tasks

    all_tasks = prepare_tasks(train_pairs, 'train') + \
                prepare_tasks(val_pairs, 'val') + \
                prepare_tasks(test_pairs, 'test')

    # دالة النقل السريع
    def copy_pair(task):
        src_img, dst_img, src_lbl, dst_lbl = task
        shutil.copyfile(src_img, dst_img)
        shutil.copyfile(src_lbl, dst_lbl)
        return True

    # 🚀 تشغيل المعالجة المتعددة بأقصى طاقة
    workers = 32
    CONSOLE.print(f"🚀 جاري تقسيم الداتا باستخدام {workers} Workers لسرعة فائقة...")

    with ProcessPoolExecutor(max_workers=workers) as executor:
        futures = [executor.submit(copy_pair, task) for task in all_tasks]
        for _ in tqdm(as_completed(futures), total=len(all_tasks), desc="Distributing Files"):
            pass

    # أسماء الكلاسات الـ 16 المعتمدة لإنشاء ملف الربط
    FINAL_CLASSES = [
        'Detergent', 'Cooking_Oil', 'Pasta_Noodles', 'Tea_Coffee', 'Soda',
        'Milk', 'Cheese', 'Juice', 'Chocolate_Candy', 'Chips', 'Tomato_Paste',
        'Yogurt', 'Cereal_Biscuits', 'Eggs', 'Water', 'Sauces_Jam'
    ]

    # إنشاء ملف data.yaml اللي الموديل هيقرأ منه
    yaml_path = FINAL_DATASET_DIR / "data.yaml"
    yaml_data = {
        'path': str(FINAL_DATASET_DIR),
        'train': 'train/images',
        'val': 'val/images',
        'test': 'test/images',
        'nc': len(FINAL_CLASSES),
        'names': FINAL_CLASSES
    }

    with open(yaml_path, 'w', encoding='utf-8') as f:
        yaml.dump(yaml_data, f, sort_keys=False)

    # طباعة ملخص التقسيم
    _split_table = Table(title="📊 DATASET SPLIT SUMMARY", header_style="bold green")
    _split_table.add_column("Split (القسم)")
    _split_table.add_column("Count (عدد الصور)", style="bold")
    _split_table.add_row("Train (تدريب 80%)", str(len(train_pairs)))
    _split_table.add_row("Validation (تقييم 10%)", str(len(val_pairs)))
    _split_table.add_row("Test (اختبار 10%)", str(len(test_pairs)))
    _split_table.add_row("Total (الإجمالي)", str(total), style="bold cyan")
    CONSOLE.print(_split_table)

    CONSOLE.print(f"\n✅ YOLO `data.yaml` created successfully at: {yaml_path}")
    CONSOLE.print("[bold yellow]🚀 أنت الآن جاهز تماماً لتشغيل (Cell 9) وبدء التدريب![/bold yellow]")

🚀 STAGE 3 — DATASET SPLITTING: Train / Val / Test

✅ تم العثور على 8778 صورة نظيفة وجاهزة للتقسيم.

🚀 جاري تقسيم الداتا باستخدام 32 Workers لسرعة فائقة...

Distributing Files: 100%|██████████| 8778/8778 [00:05<00:00, 1543.90it/s]


           📊 DATASET SPLIT SUMMARY           
┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Split (القسم)          ┃ Count (عدد الصور) ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Train (تدريب 80%)      │ 7022              │
│ Validation (تقييم 10%) │ 878               │
│ Test (اختبار 10%)      │ 878               │
│ Total (الإجمالي)       │ 8778              │
└────────────────────────┴───────────────────┘

✅ YOLO `data.yaml` created successfully at: /content/yolo_dataset/data.yaml

🚀 أنت الآن جاهز تماماً لتشغيل (Cell 9) وبدء التدريب!

In [ ]:
# =====================================================================
# CELL 8.5 | BACKUP FINAL DATASET TO GOOGLE DRIVE (FAST ZIP METHOD)
# =====================================================================
import os
import shutil
from pathlib import Path
from rich.console import Console

console = Console()

# مسارات الداتا
FINAL_DATASET_DIR = Path("/content/yolo_dataset")
DRIVE_BACKUP_DIR = Path("/content/drive/MyDrive/NTI tasks/AI Shopping Assistant/Roboflow_Data")
DRIVE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)

# مسار الملف المضغوط
ZIP_FILE_PATH = "/content/yolo_dataset_backup.zip"
FINAL_DRIVE_ZIP = DRIVE_BACKUP_DIR / "yolo_dataset_backup.zip"

if not FINAL_DATASET_DIR.exists():
    console.print("[bold red]❌ خطأ: مجلد الداتا (yolo_dataset) غير موجود. تأكد من تشغيل Cell 8 أولاً.[/bold red]")
else:
    console.print(f"[bold cyan]📦 جاري ضغط الداتا بالكامل لتسريع النقل (Zip)...[/bold cyan]")
    console.print("[dim]ملاحظة: نقل آلاف الصور للدرايف مباشرة بيستغرق ساعات، الضغط بيخلصها في ثواني![/dim]")

    # ضغط الفولدر بالكامل
    shutil.make_archive(ZIP_FILE_PATH.replace('.zip', ''), 'zip', FINAL_DATASET_DIR)

    console.print(f"[bold yellow]🚚 جاري نقل النسخة الاحتياطية إلى Google Drive...[/bold yellow]")

    # نقل الملف للدرايف
    shutil.copyfile(ZIP_FILE_PATH, FINAL_DRIVE_ZIP)

    # مسح النسخة المضغوطة من مساحة الكولاب المؤقتة لتوفير المساحة للتدريب
    os.remove(ZIP_FILE_PATH)

    console.print(f"\n[bold green]✅ تم حفظ الداتا النهائية بأمان تام في الدرايف![/bold green]")
    console.print(f"📁 مسار النسخة الاحتياطية: {FINAL_DRIVE_ZIP}")
    console.print("[bold magenta]🚀 الداتا متأمنة.. انطلق وشغل (Cell 9) عشان الموديل يبدأ التدريب![/bold magenta]")

📦 جاري ضغط الداتا بالكامل لتسريع النقل (Zip)...

ملاحظة: نقل آلاف الصور للدرايف مباشرة بيستغرق ساعات، الضغط بيخلصها في ثواني!

KeyboardInterrupt: 

In [ ]:
# =====================================================================
# CELL 9 | YOLO11 NANO (SUPER TUNED) + SMART MONITOR
# =====================================================================
import os
from pathlib import Path
from IPython.display import clear_output, Image, display
import torch
from rich.console import Console
from rich.panel import Panel

console = Console()

# 1. تحديث مكتبة Ultralytics
console.print("⏳ Updating 'ultralytics' to ensure YOLO11 support...", style="bold cyan")
os.system("pip install -q --upgrade ultralytics")
clear_output()

from ultralytics import YOLO
import cv2
cv2.imshow = lambda *args: None

# 2. إعداد المسارات
DATA_YAML = "/content/yolo_dataset/data.yaml"
DRIVE_RUNS_DIR = Path("/content/drive/MyDrive/NTI tasks/AI Shopping Assistant/Training_Runs")
DRIVE_RUNS_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# 🧠 الدالة الذكية (Smart Monitor Callback)
# ---------------------------------------------------------------------
class SmartYOLOMonitor:
    def __init__(self):
        self.history = []
        self.overfit_counter = 0
        self.best_map = 0.0

    def check_status(self, trainer):
        epoch = trainer.epoch + 1
        metrics = trainer.metrics

        try:
            train_loss = float(trainer.tloss.sum()) if hasattr(trainer, 'tloss') else 0.0
            val_loss = sum(v for k, v in metrics.items() if 'val/' in k and 'loss' in k)
            map50 = next((v for k, v in metrics.items() if 'mAP50' in k and '95' not in k), 0.0)
        except:
            return

        status_msg = "[bold green]✅ الموديل يتعلم بشكل ممتاز ومستقر[/bold green]"

        if len(self.history) > 0:
            prev_val_loss = self.history[-1]['val_loss']
            prev_train_loss = self.history[-1]['train_loss']

            if val_loss > prev_val_loss and train_loss < prev_train_loss:
                self.overfit_counter += 1
            else:
                self.overfit_counter = max(0, self.overfit_counter - 1)

        if self.overfit_counter >= 3:
            status_msg = "[bold red]⚠️ تحذير: احتمال Overfitting! الموديل يحفظ البيانات بدلاً من فهمها.[/bold red]"

        if map50 > self.best_map:
            self.best_map = map50
            if map50 > 0.85:
                status_msg = "[bold magenta]🔥 أداء جبار! الدقة تعدت 85% وما زالت في تحسن![/bold magenta]"
            elif map50 > 0.60:
                status_msg = "[bold cyan]🚀 أداء ممتاز! الموديل يكسر أرقاماً جديدة في الدقة![/bold cyan]"

        self.history.append({'train_loss': train_loss, 'val_loss': val_loss, 'map50': map50})

        report = (
            f"🎯 [bold]دقة الموديل (mAP):[/bold] {map50 * 100:.2f}%\n"
            f"📉 [bold]معدل الخطأ (Train Loss):[/bold] {train_loss:.4f} | [bold](Val Loss):[/bold] {val_loss:.4f}\n"
            f"💡 [bold]تشخيص الموديل:[/bold] {status_msg}"
        )
        console.print(Panel(report, title=f"📊 تقرير الذكاء الاصطناعي | Epoch {epoch}", expand=False, border_style="blue"))

monitor = SmartYOLOMonitor()
# ---------------------------------------------------------------------

if not os.path.exists(DATA_YAML):
    console.print("[bold red]❌ CRITICAL ERROR: data.yaml not found! Please run Cell 8 first.[/bold red]")
else:
    # استخدام نسخة Nano الخفيفة والسريعة جداً
    console.print("\n⏳ Loading YOLO11 Nano model (yolo11n.pt)...", style="bold yellow")
    model = YOLO("yolo11n.pt")
    model.add_callback("on_fit_epoch_end", monitor.check_status)

    device_type = 0 if torch.cuda.is_available() else 'cpu'
    if torch.cuda.is_available():
        console.print(f"🔥 GPU Detected: {torch.cuda.get_device_name(0)}", style="bold green")

    console.print(f"\n⚡ Starting Super-Tuned Nano Training...", style="bold cyan")

    try:
        results = model.train(
            data=DATA_YAML,
            epochs=50,
            imgsz=640,
            batch=32,                    # 🔥 باتش كبير عشان النانو يقرن صور كتير ببعض
            workers=16,                  # 📉 قللنا العمال النص عشان الرامات متعملش كراش تاني
            cache=False,                 # ⛔ قفلنا الكاش عشان نحمي الكولاب من الانفجار
            cos_lr=True,                 # 🧠 تفعيل النزول الذكي لمعدل التعلم
            close_mosaic=10,             # 🎯 تركيز عالي في آخر 10 لفات
            project=str(DRIVE_RUNS_DIR),
            name="yolo11n_supertuned_v4",
            exist_ok=True,
            patience=15,
            device=device_type,
            save=True,
            plots=True
        )
    except RuntimeError as e:
        if "Out of memory" in str(e) or "CUDA out of memory" in str(e):
            console.print("\n⚠️ ذاكرة كارت الشاشة امتلأت! جاري المحاولة بـ Batch=16 ...", style="bold yellow")
            torch.cuda.empty_cache()
            results = model.train(
                data=DATA_YAML, epochs=100, imgsz=640, batch=16, workers=16, cache=False,
                cos_lr=True, close_mosaic=10,
                project=str(DRIVE_RUNS_DIR), name="yolo11n_supertuned_v4", exist_ok=True,
                patience=15, device=device_type, save=True, plots=True
            )
        else:
            raise e

    console.print(f"\n🎉 SUCCESS! Super-Tuned Nano Training Complete.", style="bold green")
    console.print(f"✅ Your model is saved at: {DRIVE_RUNS_DIR / 'yolo11n_supertuned_v4'}")

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


⏳ Loading YOLO11 Nano model (yolo11n.pt)...

🔥 GPU Detected: Tesla T4

⚡ Starting Super-Tuned Nano Training...

Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11n_supertuned_v4, nbs=64, 

🎉 SUCCESS! Super-Tuned Nano Training Complete.

✅ Your model is saved at: /content/drive/MyDrive/NTI tasks/AI Shopping 
Assistant/Training_Runs/yolo11n_supertuned_v4

In [ ]:
# =====================================================================
# CELL 10 | FINAL EXPORT & DEPLOYMENT PACKAGE
# =====================================================================
import os
import shutil
from pathlib import Path
from rich.console import Console
from rich.table import Table

console = Console()

# 1. المسارات الأساسية
DRIVE_RUNS_DIR = Path("/content/drive/MyDrive/NTI tasks/AI Shopping Assistant/Training_Runs")
DEPLOYMENT_DIR = Path("/content/drive/MyDrive/NTI tasks/AI Shopping Assistant/Deployment_Ready")
DEPLOYMENT_DIR.mkdir(parents=True, exist_ok=True)

console.print("[bold cyan]🔍 جاري البحث عن أفضل موديل تم تدريبه...[/bold cyan]")

# 2. البحث عن أحدث فولدر تدريب أوتوماتيكياً
if not DRIVE_RUNS_DIR.exists():
    console.print("[bold red]❌ فولدر التدريب غير موجود![/bold red]")
else:
    # ترتيب الفولدرات حسب الأحدث
    all_runs = [d for d in DRIVE_RUNS_DIR.iterdir() if d.is_dir()]
    if not all_runs:
        console.print("[bold red]❌ لم يتم العثور على أي عمليات تدريب سابقة.[/bold red]")
    else:
        latest_run = max(all_runs, key=os.path.getmtime)
        best_weights = latest_run / "weights" / "best.pt"
        results_png = latest_run / "results.png"
        confusion_matrix = latest_run / "confusion_matrix.png"

        if not best_weights.exists():
            console.print(f"[bold red]❌ لم يتم العثور على ملف best.pt في {latest_run.name}[/bold red]")
        else:
            console.print(f"[bold green]✅ تم العثور على أفضل موديل في: {latest_run.name}[/bold green]")

            # 3. تجهيز الفولدر المؤقت للتصدير
            export_folder = Path("/content/Model_Export")
            if export_folder.exists():
                shutil.rmtree(export_folder)
            export_folder.mkdir()

            # 4. نسخ الملفات الذهبية
            console.print("[bold yellow]🚚 جاري تجميع العقل الذكي (best.pt) والنتائج...[/bold yellow]")
            shutil.copy(best_weights, export_folder / "shopping_model_best.pt")

            if results_png.exists():
                shutil.copy(results_png, export_folder / "training_results.png")
            if confusion_matrix.exists():
                shutil.copy(confusion_matrix, export_folder / "confusion_matrix.png")

            # إنشاء ملف تعليمات بسيط
            with open(export_folder / "README_DEPLOYMENT.txt", "w", encoding="utf-8") as f:
                f.write("🛒 AI Shopping Assistant - Final Model\n")
                f.write("======================================\n")
                f.write("هذا الملف يحتوي على أفضل أوزان للموديل (shopping_model_best.pt) جاهزة للاستخدام.\n")
                f.write("للـ Inference:\n")
                f.write("from ultralytics import YOLO\n")
                f.write("model = YOLO('shopping_model_best.pt')\n")
                f.write("results = model('image.jpg')\n")

            # 5. ضغط الملفات
            zip_filename = "Shopping_Assistant_Model_Final"
            zip_path = f"/content/{zip_filename}.zip"
            final_drive_zip = DEPLOYMENT_DIR / f"{zip_filename}.zip"

            console.print("[bold cyan]📦 جاري ضغط الملفات لتكون جاهزة للتحميل المباشر...[/bold cyan]")
            shutil.make_archive(f"/content/{zip_filename}", 'zip', export_folder)

            # نقل الـ ZIP للـ Drive
            shutil.copy(zip_path, final_drive_zip)

            # تنظيف الكولاب
            shutil.rmtree(export_folder)
            os.remove(zip_path)

            # 6. طباعة تقرير النجاح
            table = Table(title="🎉 تم تجهيز الموديل للإنتاج بنجاح 🎉", header_style="bold magenta")
            table.add_column("الملف", style="cyan")
            table.add_column("المحتوى", style="green")
            table.add_row("shopping_model_best.pt", "أوزان الموديل النهائية (العقل الذكي)")
            table.add_row("training_results.png", "رسم بياني يثبت كفاءة الموديل (للمناقشة)")
            table.add_row("confusion_matrix.png", "مصفوفة توضح دقة كل كلاس")
            table.add_row("README_DEPLOYMENT.txt", "أكواد تشغيل الموديل السريعة")

            console.print(table)
            console.print(f"\n[bold green]📁 مسار الملف المضغوط النهائي في الدرايف:[/bold green]")
            console.print(f"➔ {final_drive_zip}")
            console.print("\n[bold yellow]💡 نصيحة: تقدر دلوقتي تحمل ملف (Shopping_Assistant_Model_Final.zip) على جهازك، وتستخدمه في أي أبلكيشن أو تعرضه في مشروع التخرج وإنت حاطط رجل على رجل![/bold yellow]")

🔍 جاري البحث عن أفضل موديل تم تدريبه...

❌ فولدر التدريب غير موجود!

In [ ]:
# =====================================================================
# CELL 00 | THE ULTIMATE FIX FOR CV2 & NUMPY (خلية الفرمتة والإنقاذ)
# =====================================================================

print("🧹 1. جاري تنظيف بيئة كولاب من كل إصدارات cv2 و numpy المعطوبة (يرجى الانتظار)...")
!pip uninstall -y opencv-python opencv-contrib-python opencv-python-headless numpy easyocr > /dev/null 2>&1

print("⚙️ 2. جاري تسطيب الإصدارات المستقرة المتوافقة مع بعضها كالساعة...")
!pip install -q numpy==1.26.4 opencv-python-headless==4.8.0.74 easyocr ultralytics rich sentence-transformers faiss-cpu

import IPython
print("\n" + "🚀 "*15)
print("✅ تم تنظيف وتسطيب المكتبات المستقرة بنجاح!")
print("⚠️ الكولاب هيعمل (Restart) تلقائي دلوقتي حالا عشان يقرأ النضافة دي..")
print("بعد ما يعمل ريستارت، شغل خلية المحرك (Master Cell) مباشرة.")
print("🚀 "*15 + "\n")

# أمر سحري لعمل ريستارت تلقائي للـ Runtime عشان الذاكرة تنضف
IPython.Application.instance().kernel.do_shutdown(True)

🧹 1. جاري تنظيف بيئة كولاب من كل إصدارات cv2 و numpy المعطوبة (يرجى الانتظار)...
⚙️ 2. جاري تسطيب الإصدارات المستقرة المتوافقة مع بعضها كالساعة...
Reason for being yanked: deprecated, use 4.8.0.76
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
albumentations 2.0.8 requires opencv-python-headless>=4.9.0.80, but you have opencv-python-headless 4.8.0.74 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.

{'status': 'ok', 'restart': True}